In [4]:
import pandas as pd
import numpy as np
import re

In [7]:
data_path = 'event_level_data_dirty.csv'
clean_path = 'event_level_data_clean.csv'
df = pd.read_csv(data_path)
df.head()


,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,timezone_info
0,1,2025-01-01 07:01:00,2.0,7.0,False,True,1.280884,{}103.869885{},rainy,24.8,93.5,0,SEMBAWANG EATING HOUSE,NaN
1,2,2025-01-01 07:16:00,2.0,7.0,\r\nFalse\r\n,\nTrue\n,1.280884,103.869885,rainy,60.0,91.6,0,SEMBAWANG EATING HOUSE,NaN
2,3,NaN,2.0,7.0,False,NaN,1.280884,103.869885,cloudy,()23.7,86.9,0,SEMBAWANG EATING HOUSE,NaN
3,4,2025-01-01 07:38:00,2.0,7.0,@False,True,1.280884,103.869885,cloudy,24.2,85.7,0,SEMBAWANG EATING HOUSE,NaN
4,5,2025-01-01 07:39:00,NaN,7.0,\r\nFalse\r\n,True,\r\n1.280884\r\n,103.869885,rainy,24.7,91().2,0,SEMBAWANG EATING HOUSE,NaN


In [8]:
def clean_data(input_path, output_path):
    # Load data
    df = pd.read_csv(input_path)

    # Drop duplicates and unnecessary columns
    df = df.drop_duplicates(subset=['record_id'], keep='first')
    if 'timezone_info' in df.columns:
        df = df.drop(columns=['timezone_info'])

    # Clean String Noise
    def basic_clean(text):
        if pd.isna(text): return text
        text = str(text).strip().lower()
        text = re.sub(r'[{}()\[\]@*&|°\\ø$%//#/]', '', text)
        text = text.replace('á', 'a').replace('ë', 'e').replace('ï', 'i').replace('ü', 'u').replace('û', 'ou')
        return text

    for col in ['weather', 'is_weekend', 'is_public_holiday', 'location_name']:
        df[col] = df[col].apply(basic_clean)

    def clean_weather(value):
        if pd.isna(value):
            return 'unknown'
        elif 'night_clear' in value:
            return 'night_clear'
        elif 'rain' in value:
            return 'rainy'
        elif 'cloud' in value:
            return 'cloudy'
        elif 'clear' in value:
            return 'clear'
        else:
            return 'other'
    df['weather'] = df['weather'].apply(clean_weather)

    # Handle Timestamps and Date Features
    # Fill missing timestamps based on sequential record_ids
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['timestamp'] = df['timestamp'].interpolate(method='linear')

    # Recalculate time features to ensure consistency
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['is_weekend'] = df['day_of_week'].isin([5, 6])

    # Clean Numeric Columns (Lat, Long, Temp, Humidity)
    def clean_numeric(val):
        if pd.isna(val): return np.nan
        # Extract digits, dots, and minus signs only
        clean_val = re.sub(r'[^0-9.\-]', '', str(val))
        try:
            return float(clean_val)
        except:
            return np.nan

    num_cols = ['lat', 'long', 'temperature', 'humidity']
    for col in num_cols:
        df[col] = df[col].apply(clean_numeric)

    # Impute Location Data using Location ID
    # Use location_id to fix incorrect Lat/Long/Name (e.g., -100.0 or 999.0)
    for col in ['lat', 'long', 'location_name']:
        # Create a mapping of location_id to the most common (mode) valid value
        mapping = df[df[col].notna() & (df[col] != 0) & (df[col] != -100) & (df[col] != 999)] \
                    .groupby('location_id')[col].agg(lambda x: x.value_counts().index[0])
        df[col] = df['location_id'].map(mapping)

    # Handle Outliers in Environment Data
    # Cap temperatures to reasonable ranges (e.g., 20-40) or interpolate
    df.loc[(df['temperature'] < 15) | (df['temperature'] > 45), 'temperature'] = np.nan
    df['temperature'] = df.groupby('location_id')['temperature'].transform(lambda x: x.interpolate().ffill().bfill())
    df['humidity'] = df.groupby('location_id')['humidity'].transform(lambda x: x.interpolate().ffill().bfill())

    # Final Formatting
    df['is_public_holiday'] = df['is_public_holiday'].map({'true': True, 'false': False}).fillna(False).astype(bool)
    df['location_id'] = df['location_id'].astype(int)

    # Sort and Save
    df = df.sort_values('record_id')
    df.to_csv(output_path, index=False)
    return df

In [9]:
# Run the cleaning
clean_df = clean_data(data_path, clean_path)

In [10]:
clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name
0,1,2025-01-01 07:01:00,2,7,False,True,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house
1,2,2025-01-01 07:16:00,2,7,False,True,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house
2,3,2025-01-01 07:27:00,2,7,False,False,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house
3,4,2025-01-01 07:38:00,2,7,False,True,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house
4,5,2025-01-01 07:39:00,2,7,False,True,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house


In [11]:
clean_df['is_weekend'].dtype
clean_df['is_public_holiday'].dtype

dtype('bool')

In [12]:
# Changing True/False values to numeric 1/0
clean_df['is_weekend'] = clean_df['is_weekend'].astype(int)
clean_df['is_public_holiday'] = clean_df['is_public_holiday'].astype(int)

In [13]:
clean_df['weather'].unique()

<StringArray>
['rainy', 'cloudy', 'unknown', 'other', 'clear', 'night_clear']
Length: 6, dtype: str

In [14]:
# Drop unkown(nan) values from weather
clean_df = clean_df[clean_df['weather'] != 'unknown']
clean_df = clean_df[clean_df['weather'] != 'other']
clean_df['weather'].unique()

<StringArray>
['rainy', 'cloudy', 'clear', 'night_clear']
Length: 4, dtype: str

In [15]:
# Use dummy variables for weather
weather_map = {'rainy' : 0, 'cloudy': 1, 'night_clear': 2, 'clear': 3}
clean_df['weather_final'] = clean_df['weather'].map(weather_map)
clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,weather_final
0,1,2025-01-01 07:01:00,2,7,0,1,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house,0
1,2,2025-01-01 07:16:00,2,7,0,1,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house,0
2,3,2025-01-01 07:27:00,2,7,0,0,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house,1
3,4,2025-01-01 07:38:00,2,7,0,1,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house,1
4,5,2025-01-01 07:39:00,2,7,0,1,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house,0


In [16]:
# Creating location frequency column for encoding
location_freq = clean_df['location_name'].value_counts()
clean_df['location_freq'] = clean_df['location_name'].map(location_freq)
clean_df.head()

,record_id,timestamp,day_of_week,hour_of_day,is_weekend,is_public_holiday,lat,long,weather,temperature,humidity,location_id,location_name,weather_final,location_freq
0,1,2025-01-01 07:01:00,2,7,0,1,1.280884,103.869885,rainy,24.80,93.5,0,sembawang eating house,0,10213
1,2,2025-01-01 07:16:00,2,7,0,1,1.280884,103.869885,rainy,24.25,91.6,0,sembawang eating house,0,10213
2,3,2025-01-01 07:27:00,2,7,0,0,1.280884,103.869885,cloudy,23.70,86.9,0,sembawang eating house,1,10213
3,4,2025-01-01 07:38:00,2,7,0,1,1.280884,103.869885,cloudy,24.20,85.7,0,sembawang eating house,1,10213
4,5,2025-01-01 07:39:00,2,7,0,1,1.280884,103.869885,rainy,24.70,91.2,0,sembawang eating house,0,10213


In [17]:
# Reverse dictionary: frequency → list of locations
freq_to_locations = (
    clean_df[['location_name', 'location_freq']]
    .drop_duplicates()
    .groupby('location_freq')['location_name']
    .apply(list)
)

freq_to_locations.loc[10213]


['sembawang eating house']

In [ ]:
df = clean_df.copy()

print("clean_df rows:", len(df))
print("clean_df columns:", df.columns.tolist())

clean_df rows: 201391
clean_df columns: ['record_id', 'timestamp', 'day_of_week', 'hour_of_day', 'is_weekend', 'is_public_holiday', 'lat', 'long', 'weather', 'temperature', 'humidity', 'location_id', 'location_name', 'weather_final', 'location_freq']


In [26]:
# Create 5 mins time bins
df["time_bin"] = df["timestamp"].dt.floor("5min")

# Quick proof check: should only be 0,5,10,...55
print("Minutes in time_bin:", sorted(df["time_bin"].dt.minute.unique())[:12], "...")


Minutes in time_bin: [np.int32(0), np.int32(5), np.int32(10), np.int32(15), np.int32(20), np.int32(25), np.int32(30), np.int32(35), np.int32(40), np.int32(45), np.int32(50), np.int32(55)] ...


In [29]:
# Aggregate to 1 row per time bin
if "time_bin" in df.columns:
    group_cols = ["time_bin"]
    if "location_id" in df.columns:
        group_cols = ["location_id", "time_bin"]

    # Build agg rules only using columns that exist
    agg_rules = {}
    for col, rule in [
        ("hour_of_day", "first"),
        ("day_of_week", "first"),  
        ("is_weekend", "first"),
        ("is_public_holiday", "first"),
        ("temperature", "mean"),
        ("humidity", "mean"),
        ("weather", "first"),
        ("weather_final", "first"),   
        ("location_freq", "first"),   
    ]:
        if col in df.columns:
            agg_rules[col] = rule

    df5 = df.groupby(group_cols, as_index=False).agg(agg_rules)
    df5 = df5.sort_values("time_bin").reset_index(drop=True)
else:
    # No timestamp — just work at row level
    df5 = df.drop_duplicates().copy()

print("df5 rows (after optional bin+agg):", len(df5))
print(df5.head())

df5 rows (after optional bin+agg): 176243
   location_id            time_bin  hour_of_day  day_of_week  is_weekend  \
0           13 2025-01-01 01:10:00            1            2           0   
1           13 2025-01-01 01:25:00            1            2           0   
2           18 2025-01-01 01:55:00            1            2           0   
3           15 2025-01-01 02:05:00            2            2           0   
4           19 2025-01-01 02:40:00            2            2           0   

   is_public_holiday  temperature  humidity weather  weather_final  \
0                  1         24.2      97.3   rainy              0   
1                  1         24.8      88.8  cloudy              1   
2                  1         23.5      99.0   rainy              0   
3                  1         22.9      99.0   rainy              0   
4                  1         23.9      88.9  cloudy              1   

   location_freq  
0          10029  
1          10029  
2           9978  
3   

In [28]:
# Keep only rules for columns that actually exist
agg_rules = {k: v for k, v in agg_rules.items() if k in df.columns}

df5 = df.groupby(group_cols, as_index=False).agg(agg_rules)
df5 = df5.sort_values("time_bin").reset_index(drop=True)

print("Rows after 5-min bin + agg:", len(df5))
print(df5.head())

# PROOF it is binned: minutes should be only 0,5,10,...55
print("Minutes in df5 time_bin:", sorted(df5["time_bin"].dt.minute.unique()))


Rows after 5-min bin + agg: 176243
   location_id            time_bin  hour_of_day  day_of_week  is_weekend  \
0           13 2025-01-01 01:10:00            1            2           0   
1           13 2025-01-01 01:25:00            1            2           0   
2           18 2025-01-01 01:55:00            1            2           0   
3           15 2025-01-01 02:05:00            2            2           0   
4           19 2025-01-01 02:40:00            2            2           0   

   is_public_holiday  temperature  humidity weather  weather_final  \
0                  1         24.2      97.3   rainy              0   
1                  1         24.8      88.8  cloudy              1   
2                  1         23.5      99.0   rainy              0   
3                  1         22.9      99.0   rainy              0   
4                  1         23.9      88.9  cloudy              1   

   location_freq  
0          10029  
1          10029  
2           9978  
3          

In [32]:
# BUild feature X
num_cols = [c for c in ["day_of_week", "hour_of_day", "is_weekend", "is_public_holiday", "temperature", "humidity"]
            if c in df5.columns]

cat_cols = []
# If you still have weather text, include it as categorical (we will one-hot it later)
if "weather" in df5.columns:
    cat_cols.append("weather")

X = df5[num_cols + cat_cols].copy()

# Convert booleans to 0/1 (safe)
for bcol in ["is_weekend", "is_public_holiday"]:
    if bcol in X.columns:
        X[bcol] = X[bcol].astype(int)

print("Numeric features:", num_cols)
print("Categorical features:", cat_cols)

Numeric features: ['day_of_week', 'hour_of_day', 'is_weekend', 'is_public_holiday', 'temperature', 'humidity']
Categorical features: ['weather']


In [33]:
# Creat pseudo target y
hour = df5["hour_of_day"]
temp = df5["temperature"]
hum  = df5["humidity"]
weekend = df5["is_weekend"].astype(int) if "is_weekend" in df5.columns else 0

# Minimal safety (optional)
hour = hour.fillna(12)
temp = temp.fillna(temp.median())
hum  = hum.fillna(hum.median())

after_noon = (hour > 12).astype(int)

y = (0.2*temp + 0.1*hum + 5*weekend + 0.05*(temp*hum)*after_noon).astype(float)



In [34]:
# tim-split based 

split = int(len(df5) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print("Train size:", len(X_train), "Test size:", len(X_test))

Train size: 140994 Test size: 35249


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ],
    remainder="drop"
)


In [47]:
# Train & evalluate models
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

def evaluate(name, model):
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))   # ✅ FIXED
    r2 = r2_score(y_test, pred)

    print(f"\n=== {name} ===")
    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R2  :", round(r2, 3))

evaluate("LinearRegression", LinearRegression())
evaluate("RandomForest", RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))

try:
    from xgboost import XGBRegressor

    evaluate("XGBoost", XGBRegressor(
        n_estimators=800,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
except Exception as e:
    print("\nXGBoost not available or import failed.")
    print("Install with: pip install xgboost")
    print("Error:", e)




=== LinearRegression ===
MAE : 30.135
RMSE: 63.404
R2  : 0.664

=== RandomForest ===
MAE : 0.061
RMSE: 0.376
R2  : 1.0

=== XGBoost ===
MAE : 2.745
RMSE: 29.596
R2  : 0.927
